<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# Aim
The purpose of this notebook is present how AITune stores and loads checkpoints.

Basically, storing and loading tuned models or pipelines consists of two parts:
- checkpointing mechanism - which uses torch `state_dict` to dump and load weights
- storage mechanism - which is used to write the resulting artifacts to disk.

This notebook presents checkpointing. The storage is presented in `storage.ipynb` notebook.

In [ ]:
%%capture
%cd ..

In [ ]:
import copy
import os
from logging import basicConfig

import torch
import torch.nn as nn

from aitune.torch.backend import TorchInductorJitBackend
from aitune.torch.module_registry import MODULE_REGISTRY
from aitune.torch import load, save, tune, LocalTorchStorage, OneBackendStrategy, Module

In [ ]:
log_level = os.environ.get("AITUNE_LOG_LEVEL", "INFO")
basicConfig(level=log_level, format="%(asctime)s - %(levelname)s - %(message)s", force=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Simple toy model
Let's start with something simple.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out

In [ ]:
test_data = torch.randn(2, 5, device=device)
tune_data = torch.randn(5, device=device)

### Let's wrap module the whole model
Below we are wrapping the whole model i.e. top torch module.


In [ ]:
model = SimpleNet(5, 10, 2).to(device)
original_model = copy.deepcopy(model)  # let's keep original model for comparison
module = Module(model, "demo-simple", strategy=OneBackendStrategy(TorchInductorJitBackend()))

Let's tune the module and save it.

In [ ]:
tune(module, tune_data, batch_sizes=[1, 2], dry_run=False)

### Let's save it

In [ ]:
storage = LocalTorchStorage(base_folder="notebooks/checkpoints")
save(module, "demo-simple.ait", storage=storage)

Saving works similar to torch i.e. first, module is serialized to `state_dict` and then stored. You can customize storage with `LocalTorchStorage` class. If you don't provide `storage` by default all checkpoints will be saved to `checkpoints` folder.

Note: you can find more information regarding storage in `storage.ipynb` notebook

Let's see original model and tuned one.

#### Original model
Original model stores parameters for each layer.

In [ ]:
SimpleNet(5, 10, 2).state_dict().keys()

#### Tuned module
Basically each torch module which has been wrapped and tuned is replaced with a tuned module information.

In [ ]:
module.state_dict()[''].keys()

`tuned_module` replaced top torch module, and stored the following information:

In [ ]:
module.state_dict()['']['tuned_module'].keys()

Since this is a JIT backend, under the hood it also stores original module weights so that it can do recompilation on `load`.

The `backends` part contains mapping from `input_spec` to a backend. We have only one `graph_spec` so there is only one backend.

In [ ]:
module.state_dict()['']['tuned_module']['backends']

## Let's load module.

We don't need to have wrapping information to load the module i.e. we can do it from basic torch model.

In [ ]:
model = SimpleNet(5, 10, 2).to(device)
model_loaded = load(model, "demo-simple.ait", storage=storage)

In [ ]:
model_loaded.state

In [ ]:
model_loaded(test_data)

Let's verify that the model is loaded correctly.

In [ ]:
torch.allclose(original_model(test_data), model_loaded(test_data))

Let's clear registered modules for the next example.

In [ ]:
MODULE_REGISTRY.clear()

### Let's wrap only some layers of a model

In this example, instead of tuning the whole model, we will tune only some modules.


In [ ]:
model = SimpleNet(5, 10, 2).to(device)
original_model = copy.deepcopy(model)  # let's keep original model for comparison
model.fc1 = Module(model.fc1, "demo-simple1", strategy=OneBackendStrategy(TorchInductorJitBackend()))
model.fc3 = Module(model.fc3, "demo-simple2", strategy=OneBackendStrategy(TorchInductorJitBackend()))

Let's tune the module and save it.

In [ ]:
tune(model, tune_data, batch_sizes=[1, 2], dry_run=False)

### Let's save it

In [ ]:
save(model, "demo-simple2.ait", storage=storage)

Let's see tuned module state.

In [ ]:
model.state_dict().keys()

Tuned modules replaced torch modules (`fc1.`, `fc3.`), original torch modules stayed intact (`fc2.weight` and `f2.bias`).

## Let's load module.

Contrary to Model Navigator we don't need to have wrapping information to load the module. Can do it from basic torch model.

In [ ]:
model = SimpleNet(5, 10, 2).to(device)
model_loaded = load(model, "demo-simple2.ait", storage=storage)

In [ ]:
model_loaded(test_data)

Let's verify that the model is loaded correctly.

In [ ]:
torch.allclose(original_model(test_data), model_loaded(test_data))

In [ ]:
MODULE_REGISTRY.clear()

# Simple pipeline
The following is example of a simple pipeline i.e. an object which does not inherit from `torch.nn.Module`. This resembles how Hugging Face pipelines are constructed.

In [ ]:
class Pipeline:

    def __init__(self, input_size, hidden_size, num_classes):
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size, num_classes)

    def __call__(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out
    
    def to(self, device):
        self.fc1 = self.fc1.to(device)
        self.fc2 = self.fc2.to(device)
        self.fc3 = self.fc3.to(device)
        return self

In [ ]:
pipeline = Pipeline(5, 10, 2).to(device)
original_pipeline = copy.deepcopy(pipeline)  # let's keep original pipeline for comparison
pipeline.fc1 = Module(pipeline.fc1, "demo-simple", strategy=OneBackendStrategy(TorchInductorJitBackend()))
pipeline.fc3 = Module(pipeline.fc3, "demo-simple2", strategy=OneBackendStrategy(TorchInductorJitBackend()))

### Let's tune


In [ ]:
tune(pipeline, tune_data, batch_sizes=[1, 2], dry_run=False)

### Let's save it


In [ ]:
save(pipeline, "demo-simple-pipeline.ait", storage=storage)

Let's see pipeline state dict.

In [ ]:
torch.load("notebooks/checkpoints/demo-simple-pipeline/state_dict.pt").keys()

## Let's load pipeline.

In [ ]:
pipeline = Pipeline(5, 10, 2).to(device)
# normally pipeline would load same weights for fc2 - we have to simulate that
pipeline.fc2 = original_pipeline.fc2
# rest of layers will be loaded from checkpoint
pipeline_loaded = load(pipeline, "demo-simple-pipeline.ait", storage=storage)

In [ ]:
pipeline_loaded(test_data)

Let's verify that the pipeline is loaded correctly.

In [ ]:
torch.allclose(original_pipeline(test_data), pipeline_loaded(test_data))

In [ ]:
MODULE_REGISTRY.clear()

# Complex pipeline

Very often pipelines contain various child objects like torch modules, tokenizers, configuration etc.

AITune supports saving and loading tuned modules of a pipeline with a restriction that they must be direct children of a pipeline. 
If you have more complex case like tuning child of child torch module (i.e. grandchild of a pipeline) you have to `save/load` them one by one.

This limitation is due to the fact that if you don't save `torch.nn.Module` or wrapped module as a top module (like in case of pipelines) it is hard to track inner children back to a parent and avoid ambiguity.

Below is an example of such case.

In [ ]:
class Pipeline:
    def __init__(self, input_size, hidden_size, num_classes):
        self.net = SimpleNet(input_size, hidden_size, num_classes)
        self.preprocessor = lambda x: 2 * x
        self.postprocessor = lambda x: x + 1

    def __call__(self, x):
        x = self.preprocessor(x)
        x = self.net(x)
        x = self.postprocessor(x)
        return x
    
    def to(self, device):
        self.net = self.net.to(device)
        return self

In [ ]:
pipeline = Pipeline(5, 10, 2).to(device)
original_pipeline = copy.deepcopy(pipeline)  # let's keep original pipeline for comparison

Let's tune children of a net torch module.

In [ ]:

pipeline.net.fc1 = Module(pipeline.net.fc1, "demo-pipecomplex-fc1", strategy=OneBackendStrategy(TorchInductorJitBackend()))
pipeline.net.fc2 = Module(pipeline.net.fc2, "demo-pipecomplex-fc2", strategy=OneBackendStrategy(TorchInductorJitBackend()))
pipeline.net.fc3 = Module(pipeline.net.fc3, "demo-pipecomplex-fc3", strategy=OneBackendStrategy(TorchInductorJitBackend()))

Let's tune the pipeline and save it.


In [ ]:
tune(pipeline, tune_data, batch_sizes=[1, 2], dry_run=False)

### Let's save it
We have to save each gran child tuned object separately.


In [ ]:
save(pipeline.net.fc1, "demo-pipecomplex-fc1.ait", storage=storage)
save(pipeline.net.fc2, "demo-pipecomplex-fc2.ait", storage=storage)
save(pipeline.net.fc3, "demo-pipecomplex-fc3.ait", storage=storage)

## Let's load pipeline.
When loading, similarly to saving, we have to do it for each tuned module.

In [ ]:
pipeline = Pipeline(5, 10, 2).to(device)
pipeline.net.fc1 = load(pipeline.net.fc1, "demo-pipecomplex-fc1.ait", storage=storage)
pipeline.net.fc2 = load(pipeline.net.fc2, "demo-pipecomplex-fc2.ait", storage=storage)
pipeline.net.fc3 = load(pipeline.net.fc3, "demo-pipecomplex-fc3.ait", storage=storage)

In [ ]:
pipeline(test_data)

Let's verify that the pipeline is loaded correctly.

In [ ]:
torch.allclose(original_pipeline(test_data), pipeline(test_data))